# Verificação rápida dos dados

In [11]:
from IPython.display import display
import duckdb
import pandas as pd
import os

PARQUET_BRONZE_DIR = "../data/02-bronze"
PARQUET_SILVER_DIR = "../data/03-silver"
PARQUET_GOLD_DIR = "../data/04-gold"

OUTPUT_DIR = "../data/05-output/notebooks"

In [ ]:
# =========================================================
# Funções auxiliares
# =========================================================

def get_parquet_files(dir_path):
    """Retorna uma lista de arquivos Parquet em um diretório específico."""

    if not os.path.exists(dir_path):
        return []

    return [
        os.path.join(dir_path, x)
        for x in os.listdir(dir_path)
        if x.endswith(".parquet")
    ]

def verificar_dados(parquet_file_names):
    """
    Lê arquivos Parquet completos para métricas
    e gera uma amostra aleatória para visualização.
    """

    resultados = {}

    for file in parquet_file_names:

        print(f"\nArquivo: {file}")

        # =====================================================
        # DataFrame completo
        # =====================================================

        df_full = duckdb.sql(
            f"""
            SELECT *
            FROM read_parquet('{file}')
            """
        ).df()

        # =====================================================
        # Amostra aleatória
        # =====================================================

        df_sample = duckdb.sql(
            f"""
            SELECT *
            FROM read_parquet('{file}')
            USING SAMPLE 10 ROWS
            """
        ).df()

        print("\nShape:", df_full.shape)

        print("\nColunas:")
        print(df_full.dtypes)

        print("\nAmostra aleatória:")
        display(df_sample)

        resultados[file] = {
            "full": df_full,
            "sample": df_sample
        }

    return resultados

def analisar_estatisticas(df):
    """
    Gera análises estatísticas básicas separando
    colunas numéricas e categóricas.
    """

    # =====================================================
    # Colunas numéricas
    # =====================================================

    df_numerico = df.select_dtypes(include=["number"])

    if not df_numerico.empty:

        print("\n" + "=" * 80)
        print("ESTATÍSTICAS - COLUNAS NUMÉRICAS")
        print("=" * 80)

        stats_numericas = df_numerico.describe().T

        stats_numericas["null_count"] = df_numerico.isnull().sum()

        stats_numericas["null_percent"] = (
            df_numerico.isnull().sum() / len(df_numerico)
        ) * 100

        display(stats_numericas)

    else:
        print("\nNenhuma coluna numérica encontrada.")

    # =====================================================
    # Colunas categóricas
    # =====================================================

    df_categorico = df.select_dtypes(
        include=["object", "category", "string"]
    )

    if not df_categorico.empty:

        print("\n" + "=" * 80)
        print("ESTATÍSTICAS - COLUNAS CATEGÓRICAS")
        print("=" * 80)

        stats_categoricas = []

        for col in df_categorico.columns:

            stats_categoricas.append({
                "coluna": col,
                "nulos": df_categorico[col].isnull().sum(),
                "%_null": round(
                    (df_categorico[col].isnull().sum() / len(df)) * 100,
                    2
                ),
                "valores_unicos": df_categorico[col].nunique(),
                "valor_mais_frequente": (
                    df_categorico[col].mode().iloc[0]
                    if not df_categorico[col].mode().empty
                    else None
                ),
                "frequencia_topo": (
                    df_categorico[col].value_counts().iloc[0]
                    if not df_categorico[col].value_counts().empty
                    else None
                )
            })

        stats_categoricas = pd.DataFrame(stats_categoricas)

        display(stats_categoricas)

    else:
        print("\nNenhuma coluna categórica encontrada.")

def salvar_output_txt(resultados, output_dir, camada):

    os.makedirs(output_dir, exist_ok=True)

    map_camada = {
        "bronze": "02-BRONZE",
        "silver": "03-SILVER",
        "gold": "04-GOLD"
    }

    for file_path, data in resultados.items():

        df = data["full"]
        df_sample = data["sample"]

        parquet_name = os.path.basename(file_path)

        txt_name = parquet_name.replace(".parquet", ".txt")

        output_file = os.path.join(
            output_dir,
            f"{map_camada[camada]}_{txt_name}"
        )

        null_counts = df.isnull().sum()

        with open(output_file, "w", encoding="utf-8") as f:

            f.write("=" * 100 + "\n")
            f.write(f"ARQUIVO: {parquet_name}\n")
            f.write("=" * 100 + "\n\n")

            # =================================================
            # Shape
            # =================================================

            f.write("SHAPE\n")
            f.write("-" * 100 + "\n")
            f.write(f"{df.shape}\n\n")

            # =================================================
            # Colunas / Tipos / Nulos
            # =================================================

            f.write("COLUNAS / TIPOS / NULOS\n")
            f.write("-" * 100 + "\n")

            for col in df.columns:

                null_count = null_counts[col]
                null_percent = (null_count / len(df)) * 100

                f.write(
                    f"{col:<30} | "
                    f"Tipo: {str(df[col].dtype):<15} | "
                    f"Nulos: {null_count:<10} | "
                    f"% Null: {null_percent:>6.2f}%\n"
                )

            # =================================================
            # Sample
            # =================================================

            f.write("\nAMOSTRA ALEATÓRIA\n")
            f.write("-" * 100 + "\n")
            f.write(df_sample.to_string(index=False))
            f.write("\n")

        print(f"TXT salvo em: {output_file}")

## Visualiza todos os dados

In [23]:
# =========================================================
# Execução
# =========================================================

PARQUET_BRONZE_FILE_NAME = get_parquet_files(PARQUET_BRONZE_DIR)
PARQUET_SILVER_FILE_NAME = get_parquet_files(PARQUET_SILVER_DIR)
PARQUET_GOLD_FILE_NAME = get_parquet_files(PARQUET_GOLD_DIR)


print("="*50, "\n\t Verificando arquivos BRONZE \n", "="*50)
bronze_resultados = verificar_dados(PARQUET_BRONZE_FILE_NAME)
salvar_output_txt(bronze_resultados, OUTPUT_DIR, "bronze")


print("\n\n", "="*50, "\n\t Verificando arquivos SILVER \n", "="*50)
silver_resultados = verificar_dados(PARQUET_SILVER_FILE_NAME)
salvar_output_txt(silver_resultados, OUTPUT_DIR, "silver")


print("\n\n", "="*50, "\n\t Verificando arquivos GOLD \n", "="*50)
gold_resultados = verificar_dados(PARQUET_GOLD_FILE_NAME)
salvar_output_txt(gold_resultados, OUTPUT_DIR, "gold")

	 Verificando arquivos BRONZE 

Arquivo: ../data/02-bronze/despesa.parquet

Shape: (196033, 30)

Colunas:
DT_GERACAO                 str
HH_GERACAO                 str
AA_EXERCICIO               str
TP_DESPESA                 str
CD_TP_ESFERA_PARTIDARIA    str
DS_TP_ESFERA_PARTIDARIA    str
SG_UF                      str
CD_MUNICIPIO               str
NM_MUNICIPIO               str
NR_ZONA                    str
NR_CNPJ_PRESTADOR_CONTA    str
SG_PARTIDO                 str
NM_PARTIDO                 str
CD_TP_DOCUMENTO            str
DS_TP_DOCUMENTO            str
NR_DOCUMENTO               str
AA_AIDF                    str
NR_AIDF                    str
CD_TP_FORNECEDOR           str
DS_TP_FORNECEDOR           str
NR_CPF_CNPJ_FORNECEDOR     str
NM_FORNECEDOR              str
DS_GASTO                   str
DT_PAGAMENTO               str
VR_GASTO                   str
VR_PAGAMENTO               str
VR_DOCUMENTO               str
CD_FONTE_DESPESA           str
DS_FONTE_DESPESA          

,DT_GERACAO,HH_GERACAO,AA_EXERCICIO,TP_DESPESA,CD_TP_ESFERA_PARTIDARIA,DS_TP_ESFERA_PARTIDARIA,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,NR_ZONA,...,NR_CPF_CNPJ_FORNECEDOR,NM_FORNECEDOR,DS_GASTO,DT_PAGAMENTO,VR_GASTO,VR_PAGAMENTO,VR_DOCUMENTO,CD_FONTE_DESPESA,DS_FONTE_DESPESA,SQ_DESPESA
0,25/04/2026,16:26:12,2025,#NULO#,4,Municipal,RS,8935,TRAMANDAÍ,-1,...,#NULO#,#NULO#,#NULO#,,0,0,0,-1,#NULO#,-1
1,25/04/2026,16:23:25,2025,D,0,Nacional,BR,-1,#NULO#,-1,...,51796316000172,Direção Municipal/Comissão Provisória - PT - F...,TRANSFERÊNCIAS FINANCEIRAS EFETUADAS - OUTROS ...,08/09/2025,"50,06","50,06","50,06",2,Outros Recursos,-1
2,25/04/2026,16:23:25,2025,D,0,Nacional,BR,-1,#NULO#,-1,...,80892029000164,Direção Municipal/Comissão Provisória - PT - M...,TRANSFERÊNCIAS FINANCEIRAS EFETUADAS - OUTROS ...,13/05/2025,"130,81","130,81","130,81",2,Outros Recursos,-1
3,25/04/2026,16:23:25,2025,D,0,Nacional,BR,-1,#NULO#,-1,...,43336767000107,Direção Municipal/Comissão Provisória - PT - S...,TRANSFERÊNCIAS FINANCEIRAS EFETUADAS - OUTROS ...,13/05/2025,"884,01","884,01","884,01",2,Outros Recursos,-1
4,25/04/2026,16:23:25,2025,D,0,Nacional,BR,-1,#NULO#,-1,...,08963082000181,Direção Estadual/Distrital - PL - RONDÔNIA,TRANSFERÊNCIAS FINANCEIRAS EFETUADAS - FUNDO P...,04/04/2025,40000,40000,40000,1,Fundo Partidário,-1
5,25/04/2026,16:23:25,2025,D,0,Nacional,BR,-1,#NULO#,-1,...,25173657000181,Direção Municipal/Comissão Provisória - NOVO -...,TRANSFERÊNCIAS FINANCEIRAS EFETUADAS - OUTROS ...,17/10/2025,6922,6922,6922,2,Outros Recursos,-1
6,25/04/2026,16:23:25,2025,G,1,Distrital,DF,-1,#NULO#,-1,...,15754475000140,HostGator Brasil Ltda,TELECOMUNICAÇÕES E INTERNET - ORDINÁRIAS,11/06/2025,"655,79","655,79","655,79",1,Fundo Partidário,3940310
7,25/04/2026,16:23:25,2025,G,1,Distrital,DF,-1,#NULO#,-1,...,82878889134,Joana DArc Gomes Cardoso,PESSOAL - SALÁRIOS E ORDENADOS - ORDINÁRIAS,03/02/2025,"3908,02","3908,02","3908,02",1,Fundo Partidário,3563840
8,25/04/2026,16:23:25,2025,G,2,Estadual,GO,-1,#NULO#,-1,...,00000000000191,BANCO DO BRASIL S.A.,DESPESAS FINANCEIRAS - COMISSÕES E TARIFAS BAN...,10/02/2025,"73,8","73,8","73,8",1,Fundo Partidário,3928918
9,25/04/2026,16:23:25,2025,G,2,Estadual,PE,-1,#NULO#,-1,...,50172126000111,50.172.126 RODRIGO PESSOA BARBOZA DE ANDRADE,SERVIÇOS TÉCNICO-PROFISSIONAIS - OUTROS SERVIÇ...,06/01/2025,3000,3000,3000,1,Fundo Partidário,3580005



Arquivo: ../data/02-bronze/classificacao_despesa.parquet

Shape: (160, 2)

Colunas:
DESPESA                                                                                                                                       str
CLASSIFICACAO                                                                                                                                 str
dtype: object

Amostra aleatória:


,DESPESA,CLASSIFICACAO
0,DESPESAS FINANCEIRAS - OUTRAS DESPESAS FINANCE...,ADMINISTRATIVO
1,SERVICOS TECNICO-PROFISSIONAIS - SERVICOS CONT...,ADMINISTRATIVO
2,PESSOAL - SALARIOS E ORDENADOS - ORDINARIAS ...,ADMINISTRATIVO
3,TRIBUTOS - OUTROS TRIBUTOS - ORDINARIAS ...,ADMINISTRATIVO
4,PESSOAL - O SALARIO - ORDINARIAS ...,ADMINISTRATIVO
5,"TRANSPORTES E VIAGENS - COMBUSTIVEIS, OLEOS E ...",INDEFINIDO
6,PRODUCAO DE AUDIOVISUAIS - MULHERES ...,FINALÍSTICO
7,"TRANSPORTES E VIAGENS - COMBUSTIVEIS, OLEOS E ...",FINALÍSTICO
8,TRANSPORTES E VIAGENS - OUTRAS DESPESAS COM TR...,FINALÍSTICO
9,SERVICOS TECNICO-PROFISSIONAIS - SERVICOS DE C...,FINALÍSTICO



Arquivo: ../data/02-bronze/receita.parquet

Shape: (194844, 36)

Colunas:
DT_GERACAO                        str
HH_GERACAO                        str
CD_TP_ESFERA_PARTIDARIA           str
DS_TP_ESPERA_PARTIDARIA           str
SG_UF                             str
CD_MUNICIPIO                      str
NM_MUNICIPIO                      str
NR_ZONA                           str
NR_CNPJ_PRESTADOR_CONTA           str
SG_PARTIDO                        str
NM_PARTIDO                        str
CD_TP_ORIGEM_DOACAO               str
DS_TP_ORIGEM_DOACAO               str
NR_CPF_CNPJ_DOADOR                str
NM_DOADOR                         str
CD_TP_ESFERA_PARTIDARIA_DOADOR    str
DS_TP_ESFERA_PARTIDARIA_DOADOR    str
SG_UF_DOADOR                      str
CD_MUNICIPIO_DOADOR               str
NM_MUNICIPIO_DOADOR               str
NR_ZONA_DOADOR                    str
SQ_CANDIDATO_DOADOR               str
NR_CANDIDATO_DOADOR               str
CD_CANDIDATO_CARGO_DOADOR         str
DS_CANDIDATO_

...

TXT salvo em: ../data/05-output/notebooks/03-SILVER_despesa.txt
TXT salvo em: ../data/05-output/notebooks/03-SILVER_classificacao_despesa.txt
TXT salvo em: ../data/05-output/notebooks/03-SILVER_receita.txt
TXT salvo em: ../data/05-output/notebooks/03-SILVER_receita_enriquecida.txt
TXT salvo em: ../data/05-output/notebooks/03-SILVER_cnpj.txt
TXT salvo em: ../data/05-output/notebooks/03-SILVER_despesa_enriquecida.txt


	 Verificando arquivos GOLD 

Arquivo: ../data/04-gold/partido_ano_despesa.parquet

Shape: (34, 11)

Colunas:
sg_partido                        str
aa_exercicio                    int64
vl_despesa_total              float64
vl_despesa_administrativa     float64
vl_despesa_finalistica        float64
vl_despesa_indefinida         float64
qtd_fornecedores_unicos         int64
pct_despesa_administrativa    float64
pct_despesa_finalistica       float64
pct_despesa_indefinida        float64
ticket_medio_despesa          float64
dtype: object

Amostra aleatória:


,sg_partido,aa_exercicio,vl_despesa_total,vl_despesa_administrativa,vl_despesa_finalistica,vl_despesa_indefinida,qtd_fornecedores_unicos,pct_despesa_administrativa,pct_despesa_finalistica,pct_despesa_indefinida,ticket_medio_despesa
0,AVANTE,2025,2.188460e+07,9.806308e+06,0.0,4566747.76,277,44.809166,0.0,20.867401,79005.784260
1,CIDADANIA,2025,4.106778e+06,2.814477e+06,0.0,827583.21,288,68.532484,0.0,20.151643,14259.645451
2,MOBILIZA,2025,1.970976e+05,1.964242e+05,0.0,673.40,12,99.658342,0.0,0.341658,16424.800833
3,PC DO B,2025,2.451140e+07,1.013933e+07,0.0,4998120.07,630,41.365773,0.0,20.391003,38906.981460
4,PCO,2025,0.000000e+00,0.000000e+00,0.0,0.00,0,NaN,NaN,NaN,NaN
5,PDT,2025,5.910273e+07,2.970722e+07,0.0,16019250.72,933,50.263711,0.0,27.104080,63346.974502
6,PODE,2025,6.724362e+07,3.251337e+07,0.0,17938500.63,874,48.351602,0.0,26.676880,76937.784794
7,PP,2025,5.041076e+07,3.451222e+07,0.0,6799391.01,1213,68.462010,0.0,13.487976,41558.744938
8,PRTB,2025,0.000000e+00,0.000000e+00,0.0,0.00,0,NaN,NaN,NaN,NaN
9,PT,2025,2.261059e+08,1.086025e+08,0.0,52077272.16,3892,48.031706,0.0,23.032245,58095.049263



Arquivo: ../data/04-gold/partido_ano_receita.parquet

Shape: (34, 10)

Colunas:
sg_partido                   str
aa_exercicio               int64
vl_receita_total         float64
vl_receita_publica       float64
vl_receita_privada       float64
vl_receita_partidaria    float64
qtd_doadores_unicos        int64
pct_receita_publica      float64
pct_receita_privada      float64
ticket_medio_receita     float64
dtype: object

Amostra aleatória:


,sg_partido,aa_exercicio,vl_receita_total,vl_receita_publica,vl_receita_privada,vl_receita_partidaria,qtd_doadores_unicos,pct_receita_publica,pct_receita_privada,ticket_medio_receita
0,AGIR,2025,1.207855e+06,0.000000e+00,716425.63,4.679866e+05,56,0.000000,59.313878,2.156884e+04
1,AVANTE,2025,5.362063e+07,2.944308e+07,260244.61,1.169648e+07,41,54.909980,0.485344,1.307820e+06
2,CIDADANIA,2025,1.175145e+07,3.815505e+06,501646.01,7.429742e+06,79,32.468366,4.268799,1.487526e+05
3,PL,2025,4.354022e+08,2.086250e+08,40965912.74,1.419754e+08,741,47.915465,9.408751,5.875874e+05
4,PP,2025,1.606949e+08,5.680477e+07,3478597.68,8.376226e+07,1437,35.349449,2.164722,1.118267e+05
5,PSOL,2025,2.431282e+08,2.090511e+08,685627.79,3.338911e+07,127,85.983917,0.282003,1.914395e+06
6,PT,2025,3.664338e+08,1.529199e+08,33868524.01,1.792951e+08,26006,41.731939,9.242741,1.409036e+04
7,PTB,2025,0.000000e+00,0.000000e+00,0.00,0.000000e+00,0,NaN,NaN,NaN
8,UNIAO,2025,1.642331e+08,0.000000e+00,558854.50,1.626579e+08,334,0.000000,0.340281,4.917158e+05
9,UP,2025,6.243870e+03,0.000000e+00,6243.87,0.000000e+00,37,0.000000,100.000000,1.687532e+02


TXT salvo em: ../data/05-output/notebooks/04-GOLD_partido_ano_despesa.txt
TXT salvo em: ../data/05-output/notebooks/04-GOLD_partido_ano_receita.txt


In [4]:
from IPython.display import display
import duckdb

for file in PARQUET_SILVER_FILE_NAME:
    print(f"\nArquivo: {file}")

    df = duckdb.sql(
        f"""
        SELECT *
        FROM read_parquet('{file}')
        LIMIT 100
        """
    ).df()

    print("\nShape:", df.shape)
    print("\nColunas:")
    print(df.dtypes)

    print("\nAmostra:")
    display(df.head())


Arquivo: ../data/03-silver/classificacao_despesa.parquet

Shape: (100, 2)

Colunas:
nm_despesa    str
tp_gasto      str
dtype: object

Amostra:


,nm_despesa,tp_gasto
0,DESPESAS FINANCEIRAS - OUTRAS DESPESAS FINANCE...,ADMINISTRATIVO
1,TRANSFERENCIAS FINANCEIRAS EFETUADAS - FUNDO P...,ADMINISTRATIVO
2,TRANSFERENCIAS FINANCEIRAS EFETUADAS - OUTROS ...,ADMINISTRATIVO
3,TRANSFERENCIAS FINANCEIRAS EFETUADAS - FUNDO P...,ADMINISTRATIVO
4,TRANSFERENCIAS FINANCEIRAS EFETUADAS - FUNDO P...,FINALÍSTICO



Arquivo: ../data/03-silver/cnpj.parquet

Shape: (100, 28)

Colunas:
cd_cnpj                       str
nm_razao_social               str
nm_fantasia                   str
dt_abertura                   str
ed_uf                         str
nm_regiao_politica            str
nm_tipo_estabelecimento       str
dt_situacao_cadastral         str
nm_situacao_cadastral         str
dt_sit_especial            object
nm_situacao_especial       object
cd_natureza_juridica          str
nm_natureza_juridica          str
nm_porte                      str
vl_capital_social             str
cd_cnae                       str
in_principal                  str
cd_nivel1_secao               str
nm_nivel1_secao               str
cd_nivel2_divisao             str
nm_nivel2_divisao             str
nm_nivel3_grupo               str
cd_nivel3_grupo               str
cd_nivel4_classe              str
nm_nivel4_classe              str
cd_nivel5_subclasse           str
nm_nivel5_subclasse           str
dt_atualizaca

,cd_cnpj,nm_razao_social,nm_fantasia,dt_abertura,ed_uf,nm_regiao_politica,nm_tipo_estabelecimento,dt_situacao_cadastral,nm_situacao_cadastral,dt_sit_especial,...,nm_nivel1_secao,cd_nivel2_divisao,nm_nivel2_divisao,nm_nivel3_grupo,cd_nivel3_grupo,cd_nivel4_classe,nm_nivel4_classe,cd_nivel5_subclasse,nm_nivel5_subclasse,dt_atualizacao_pj_rfb
0,87537000130,COMERCIAL MALLET LTDA,NaN,1994-06-13 00:00:00,SC,SUL,MATRIZ,2005-11-03 00:00:00,ATIVA,None,...,COMERCIO; REPARACAO DE VEICULOS AUTOMOTORES E ...,46,"COMERCIO POR ATACADO, EXCETO VEICULOS AUTOMOTO...",COMERCIO ATACADISTA DE PRODUTOS DE CONSUMO NAO...,46.4,46.42-7,COMERCIO ATACADISTA DE ARTIGOS DO VESTUARIO E ...,46.42-7/01,COMERCIO ATACADISTA DE ARTIGOS DO VESTUARIO E ...,2026-04-13 05:46:14.493801000
1,124153000140,EMBALAGENS T 2 LTDA,NaN,1994-07-27 00:00:00,GO,CENTRO-OESTE,MATRIZ,2005-01-08 00:00:00,ATIVA,None,...,COMERCIO; REPARACAO DE VEICULOS AUTOMOTORES E ...,47,COMERCIO VAREJISTA,"COMERCIO VAREJISTA DE ARTIGOS CULTURAIS, RECRE...",47.6,47.61-0,"COMERCIO VAREJISTA DE LIVROS, JORNAIS, REVISTA...",47.61-0/03,COMERCIO VAREJISTA DE ARTIGOS DE PAPELARIA,2026-04-13 05:46:14.493801000
2,165731000197,FACILITY TOUR AGENCIA DE TURISMO LTDA,FACILITY TOUR,1994-08-16 00:00:00,SP,SUDESTE,MATRIZ,2004-04-24 00:00:00,ATIVA,None,...,ATIVIDADES ADMINISTRATIVAS E SERVICOS COMPLEME...,79,"AGENCIAS DE VIAGENS, OPERADORES TURISTICOS E S...",AGENCIAS DE VIAGENS E OPERADORES TURISTICOS,79.1,79.11-2,AGENCIAS DE VIAGENS,79.11-2/00,AGENCIAS DE VIAGENS,2026-04-13 05:46:14.493801000
3,370353000183,PARANOA HOTEIS E TURISMO LTDA,NaN,1974-01-14 00:00:00,DF,CENTRO-OESTE,MATRIZ,2005-11-03 00:00:00,ATIVA,None,...,ALOJAMENTO E ALIMENTACAO,55,ALOJAMENTO,HOTEIS E SIMILARES,55.1,55.10-8,HOTEIS E SIMILARES,55.10-8/01,HOTEIS,2026-04-13 05:46:14.493801000
4,373580000162,CYPRESS TURISMO LTDA,CYPRESS TURISMO,1994-12-30 00:00:00,RS,SUL,MATRIZ,2005-11-03 00:00:00,ATIVA,None,...,ATIVIDADES ADMINISTRATIVAS E SERVICOS COMPLEME...,79,"AGENCIAS DE VIAGENS, OPERADORES TURISTICOS E S...",AGENCIAS DE VIAGENS E OPERADORES TURISTICOS,79.1,79.11-2,AGENCIAS DE VIAGENS,79.11-2/00,AGENCIAS DE VIAGENS,2026-04-13 05:46:14.493801000



Arquivo: ../data/03-silver/despesa.parquet

Shape: (100, 30)

Colunas:
dt_geracao                 datetime64[us]
hh_geracao                            str
aa_exercicio                        int64
tp_despesa                         object
cd_tp_esfera_partidaria               str
ds_tp_esfera_partidaria               str
sg_uf                                 str
cd_municipio                          str
nm_municipio                          str
nr_zona                             int64
cd_cnpj_prestador_conta               str
sg_partido                            str
nm_partido                            str
cd_tp_documento                    object
ds_tp_documento                    object
nr_documento                       object
aa_aidf                             int64
nr_aidf                               str
cd_tp_fornecedor                   object
ds_tp_fornecedor                   object
cd_cpf_cnpj_fornecedor             object
nm_fornecedor                      object
ds_g

,dt_geracao,hh_geracao,aa_exercicio,tp_despesa,cd_tp_esfera_partidaria,ds_tp_esfera_partidaria,sg_uf,cd_municipio,nm_municipio,nr_zona,...,cd_cpf_cnpj_fornecedor,nm_fornecedor,ds_gasto,dt_pagamento,vl_gasto,vl_pagamento,vl_documento,cd_fonte_despesa,ds_fonte_despesa,sq_despesa
0,2026-04-25,16:26:12,2025,None,1,DISTRITAL,DF,-1,NaN,-1,...,None,None,None,NaT,0.0,0.0,0.0,-1,None,-1
1,2026-04-25,16:26:12,2025,None,1,DISTRITAL,DF,-1,NaN,-1,...,None,None,None,NaT,0.0,0.0,0.0,-1,None,-1
2,2026-04-25,16:26:12,2025,None,2,ESTADUAL,CE,-1,NaN,-1,...,None,None,None,NaT,0.0,0.0,0.0,-1,None,-1
3,2026-04-25,16:26:12,2025,None,2,ESTADUAL,MS,-1,NaN,-1,...,None,None,None,NaT,0.0,0.0,0.0,-1,None,-1
4,2026-04-25,16:26:12,2025,None,2,ESTADUAL,SP,-1,NaN,-1,...,None,None,None,NaT,0.0,0.0,0.0,-1,None,-1



Arquivo: ../data/03-silver/receita.parquet

Shape: (100, 38)

Colunas:
dt_geracao                        datetime64[us]
hh_geracao                                   str
cd_tp_esfera_partidaria                      str
ds_tp_espera_partidaria                      str
sg_uf                                        str
cd_municipio                              object
nm_municipio                              object
nr_zona                                  float64
cd_cnpj_prestador_conta                      str
sg_partido                                   str
nm_partido                                   str
cd_tp_origem_doacao                          str
ds_tp_origem_doacao                          str
cd_cpf_cnpj_doador                           str
nm_doador                                    str
cd_tp_esfera_partidaria_doador               str
ds_tp_esfera_partidaria_doador               str
sg_uf_doador                                 str
cd_municipio_doador                       obje

,dt_geracao,hh_geracao,cd_tp_esfera_partidaria,ds_tp_espera_partidaria,sg_uf,cd_municipio,nm_municipio,nr_zona,cd_cnpj_prestador_conta,sg_partido,...,ds_tp_natureza_recurso,cd_tp_especie_recurso,ds_tp_especie_recurso,nr_recibo_doacao,nr_documento,dt_receita,ds_receita,vl_receita,aa_exercicio,ind_dt_receita_nula
0,2026-04-25,16:24:44,0,NACIONAL,BR,None,None,NaN,32206989000180,AGIR,...,FINANCEIRO,7,TRANSFERENCIA ELETRONICA,1475,None,2025-08-05,TRANSFERENCIAS DE RECURSOS FINANCEIROS PARA MA...,2500.00,2025,False
1,2026-04-25,16:24:44,0,NACIONAL,BR,None,None,NaN,32206989000180,AGIR,...,FINANCEIRO,7,TRANSFERENCIA ELETRONICA,1490,None,2025-08-28,TRANSFERENCIAS DE RECURSOS FINANCEIROS PARA MA...,5000.00,2025,False
2,2026-04-25,16:24:44,0,NACIONAL,BR,None,None,NaN,32206989000180,AGIR,...,FINANCEIRO,7,TRANSFERENCIA ELETRONICA,1506,None,2025-10-01,TRANSFERENCIAS DE RECURSOS FINANCEIROS PARA MA...,2000.00,2025,False
3,2026-04-25,16:24:44,0,NACIONAL,BR,None,None,NaN,32206989000180,AGIR,...,FINANCEIRO,7,TRANSFERENCIA ELETRONICA,1535,None,2025-12-22,TRANSFERENCIAS DE RECURSOS FINANCEIROS PARA MA...,4000.00,2025,False
4,2026-04-25,16:24:44,0,NACIONAL,BR,None,None,NaN,32206989000180,AGIR,...,FINANCEIRO,7,TRANSFERENCIA ELETRONICA,NaN,None,2025-04-25,SOBRAS FINANCEIRAS DE CAMPANHA - CANDIDATOS,1692.07,2025,False


In [5]:
from IPython.display import display
import duckdb

for file in PARQUET_GOLD_FILE_NAME:
    print(f"\nArquivo: {file}")

    df = duckdb.sql(
        f"""
        SELECT *
        FROM read_parquet('{file}')
        LIMIT 100
        """
    ).df()

    print("\nShape:", df.shape)
    print("\nColunas:")
    print(df.dtypes)

    print("\nAmostra:")
    display(df.head())


Arquivo: ../data/04-gold/despesa_enriquecida.parquet

Shape: (100, 41)

Colunas:
dt_geracao                   datetime64[us]
hh_geracao                              str
aa_exercicio                          int64
tp_despesa                           object
cd_tp_esfera_partidaria                 str
ds_tp_esfera_partidaria                 str
sg_uf                                   str
cd_municipio                            str
nm_municipio                            str
nr_zona                               int64
cd_cnpj_prestador_conta                 str
sg_partido                              str
nm_partido                              str
cd_tp_documento                      object
ds_tp_documento                      object
nr_documento                         object
aa_aidf                               int64
nr_aidf                                 str
cd_tp_fornecedor                     object
ds_tp_fornecedor                     object
cd_cpf_cnpj_fornecedor               o

,dt_geracao,hh_geracao,aa_exercicio,tp_despesa,cd_tp_esfera_partidaria,ds_tp_esfera_partidaria,sg_uf,cd_municipio,nm_municipio,nr_zona,...,nm_fantasia,is_cnpj_enriquecido,tp_gasto,tp_classificacao_origem,in_despesa_administrativa,in_despesa_finalistica,in_despesa_indefinida,vl_despesa_administrativa,vl_despesa_finalistica,vl_despesa_indefinida
0,2026-04-25,16:26:12,2025,None,1,DISTRITAL,DF,-1,NaN,-1,...,None,False,INDEFINIDO,NAO_CLASSIFICADO,False,False,True,0.0,0.0,0.0
1,2026-04-25,16:26:12,2025,None,1,DISTRITAL,DF,-1,NaN,-1,...,None,False,INDEFINIDO,NAO_CLASSIFICADO,False,False,True,0.0,0.0,0.0
2,2026-04-25,16:26:12,2025,None,1,DISTRITAL,DF,-1,NaN,-1,...,None,False,INDEFINIDO,NAO_CLASSIFICADO,False,False,True,0.0,0.0,0.0
3,2026-04-25,16:26:12,2025,None,1,DISTRITAL,DF,-1,NaN,-1,...,None,False,INDEFINIDO,NAO_CLASSIFICADO,False,False,True,0.0,0.0,0.0
4,2026-04-25,16:26:12,2025,None,1,DISTRITAL,DF,-1,NaN,-1,...,None,False,INDEFINIDO,NAO_CLASSIFICADO,False,False,True,0.0,0.0,0.0



Arquivo: ../data/04-gold/partido_ano_despesa.parquet

Shape: (34, 11)

Colunas:
sg_partido                        str
aa_exercicio                    int64
vl_despesa_total              float64
vl_despesa_administrativa     float64
vl_despesa_finalistica        float64
vl_despesa_indefinida         float64
qtd_fornecedores_unicos         int64
pct_despesa_administrativa    float64
pct_despesa_finalistica       float64
pct_despesa_indefinida        float64
ticket_medio_despesa          float64
dtype: object

Amostra:


,sg_partido,aa_exercicio,vl_despesa_total,vl_despesa_administrativa,vl_despesa_finalistica,vl_despesa_indefinida,qtd_fornecedores_unicos,pct_despesa_administrativa,pct_despesa_finalistica,pct_despesa_indefinida,ticket_medio_despesa
0,AGIR,2025,873771.32,685084.32,0.0,188687.00,37,78.405448,0.0,21.594552,23615.441081
1,AVANTE,2025,21884602.24,9806307.71,0.0,4566747.76,277,44.809166,0.0,20.867401,79005.784260
2,CIDADANIA,2025,4106777.89,2814476.88,0.0,827583.21,288,68.532484,0.0,20.151643,14259.645451
3,DC,2025,1954621.90,1344255.70,0.0,609566.20,94,68.773183,0.0,31.185888,20793.850000
4,DEM,2025,0.00,0.00,0.0,0.00,0,NaN,NaN,NaN,NaN



Arquivo: ../data/04-gold/partido_ano_receita.parquet

Shape: (34, 10)

Colunas:
sg_partido                   str
aa_exercicio               int64
vl_receita_total         float64
vl_receita_publica       float64
vl_receita_privada       float64
vl_receita_partidaria    float64
qtd_doadores_unicos        int64
pct_receita_publica      float64
pct_receita_privada      float64
ticket_medio_receita     float64
dtype: object

Amostra:


,sg_partido,aa_exercicio,vl_receita_total,vl_receita_publica,vl_receita_privada,vl_receita_partidaria,qtd_doadores_unicos,pct_receita_publica,pct_receita_privada,ticket_medio_receita
0,AGIR,2025,1207854.99,0.00,716425.63,467986.60,56,0.000000,59.313878,2.156884e+04
1,AVANTE,2025,53620628.37,29443076.52,260244.61,11696480.12,41,54.909980,0.485344,1.307820e+06
2,CIDADANIA,2025,11751454.30,3815505.23,501646.01,7429741.60,79,32.468366,4.268799,1.487526e+05
3,DC,2025,3523606.22,0.00,1281392.84,2234784.96,150,0.000000,36.365949,2.349071e+04
4,DEM,2025,0.00,0.00,0.00,0.00,0,NaN,NaN,NaN



Arquivo: ../data/04-gold/receita_enriquecida.parquet

Shape: (100, 48)

Colunas:
dt_geracao                        datetime64[us]
hh_geracao                                   str
cd_tp_esfera_partidaria                      str
ds_tp_espera_partidaria                      str
sg_uf                                        str
cd_municipio                              object
nm_municipio                              object
nr_zona                                  float64
cd_cnpj_prestador_conta                      str
sg_partido                                   str
nm_partido                                   str
cd_tp_origem_doacao                          str
ds_tp_origem_doacao                          str
cd_cpf_cnpj_doador                           str
nm_doador                                    str
cd_tp_esfera_partidaria_doador               str
ds_tp_esfera_partidaria_doador               str
sg_uf_doador                                 str
cd_municipio_doador                 

,dt_geracao,hh_geracao,cd_tp_esfera_partidaria,ds_tp_espera_partidaria,sg_uf,cd_municipio,nm_municipio,nr_zona,cd_cnpj_prestador_conta,sg_partido,...,nm_razao_social,nm_fantasia,is_cnpj_enriquecido,tp_receita,in_receita_publica,in_receita_privada,in_receita_partidaria,vl_receita_publica,vl_receita_privada,vl_receita_partidaria
0,2026-04-25,16:24:44,0,NACIONAL,BR,None,None,NaN,32206989000180,AGIR,...,NaN,NaN,False,PARTIDARIA,False,False,True,0.0,0.0,2500.0
1,2026-04-25,16:24:44,0,NACIONAL,BR,None,None,NaN,32206989000180,AGIR,...,NaN,NaN,False,PARTIDARIA,False,False,True,0.0,0.0,2500.0
2,2026-04-25,16:24:44,0,NACIONAL,BR,None,None,NaN,32206989000180,AGIR,...,NaN,NaN,False,PARTIDARIA,False,False,True,0.0,0.0,2500.0
3,2026-04-25,16:24:44,0,NACIONAL,BR,None,None,NaN,32206989000180,AGIR,...,NaN,NaN,False,PARTIDARIA,False,False,True,0.0,0.0,2500.0
4,2026-04-25,16:24:44,0,NACIONAL,BR,None,None,NaN,32206989000180,AGIR,...,NaN,NaN,False,PARTIDARIA,False,False,True,0.0,0.0,5000.0
